In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔗 Distributed Systems & Consensus Guide

**Building reliable distributed systems with consensus algorithms, replication, and consistency models**

This guide covers:
- Consistency models (strong, eventual, causal)
- Consensus algorithms (Raft, Paxos, Byzantine)
- Distributed transactions and coordination
- Split-brain prevention
- Leader election
- Quorum-based replication
- Failure detection and recovery
- Partitioning strategies
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Consensus Algorithms

### Raft Consensus Implementation
```python
from enum import Enum
from dataclasses import dataclass
from typing import List, Dict
import random
import time
from threading import Thread, Lock

class ServerState(Enum):
    FOLLOWER = "follower"
    CANDIDATE = "candidate"
    LEADER = "leader"

@dataclass
class LogEntry:
    term: int
    command: str
    data: Dict

class RaftServer:
    """Simplified Raft consensus server"""
    
    def __init__(self, server_id: str, peers: List[str]):
        self.server_id = server_id
        self.peers = peers
        
        # Persistent state (on all servers)
        self.current_term = 0
        self.voted_for = None
        self.log: List[LogEntry] = []
        
        # Volatile state (on all servers)
        self.commit_index = 0
        self.last_applied = 0
        self.state = ServerState.FOLLOWER
        
        # Volatile state (on leaders)
        self.next_index: Dict[str, int] = {peer: len(self.log) for peer in peers}
        self.match_index: Dict[str, int] = {peer: 0 for peer in peers}
        
        # Timing
        self.election_timeout = random.uniform(150, 300) / 1000
        self.last_heartbeat = time.time()
        self.lock = Lock()
    
    def append_entry(self, command: str, data: Dict) -> bool:
        """Append entry to log"""
        with self.lock:
            if self.state != ServerState.LEADER:
                return False
            
            entry = LogEntry(term=self.current_term, command=command, data=data)
            self.log.append(entry)
            return True
    
    def request_vote(self, term: int, candidate_id: str, last_log_index: int, last_log_term: int) -> bool:
        """Handle vote request"""
        with self.lock:
            if term < self.current_term:
                return False
            
            if term > self.current_term:
                self.current_term = term
                self.voted_for = None
                self.state = ServerState.FOLLOWER
            
            # Grant vote if not voted and candidate log is at least as up-to-date
            if self.voted_for is None or self.voted_for == candidate_id:
                if self._is_log_up_to_date(last_log_index, last_log_term):
                    self.voted_for = candidate_id
                    return True
            
            return False
    
    def append_entries(self, term: int, leader_id: str, prev_log_index: int, 
                       prev_log_term: int, entries: List[LogEntry], leader_commit: int) -> bool:
        """Handle append entries (heartbeat and replication)"""
        with self.lock:
            if term < self.current_term:
                return False
            
            if term > self.current_term:
                self.current_term = term
                self.voted_for = None
            
            self.state = ServerState.FOLLOWER
            self.last_heartbeat = time.time()
            
            # Check if log contains entry at prev_log_index with term prev_log_term
            if prev_log_index >= 0 and prev_log_index >= len(self.log):
                return False
            
            if prev_log_index >= 0 and self.log[prev_log_index].term != prev_log_term:
                return False
            
            # Append entries and update commit index
            self.log = self.log[:prev_log_index + 1] + entries
            self.commit_index = min(leader_commit, len(self.log) - 1)
            
            return True
    
    def _is_log_up_to_date(self, last_log_index: int, last_log_term: int) -> bool:
        """Check if candidate log is at least as up-to-date"""
        if not self.log:
            return True
        
        last_entry = self.log[-1]
        if last_log_term > last_entry.term:
            return True
        if last_log_term == last_entry.term and last_log_index >= len(self.log) - 1:
            return True
        
        return False
    
    def check_election_timeout(self) -> bool:
        """Check if election timeout has occurred"""
        return (time.time() - self.last_heartbeat) > self.election_timeout
    
    def start_election(self) -> bool:
        """Start leader election"""
        with self.lock:
            self.current_term += 1
            self.state = ServerState.CANDIDATE
            self.voted_for = self.server_id
            self.last_heartbeat = time.time()
            
            last_log_index = len(self.log) - 1
            last_log_term = self.log[last_log_index].term if self.log else 0
            
            votes = 1  # Vote for self
            needed_votes = (len(self.peers) + 1) // 2 + 1
            
            # Send request_vote to all peers
            print(f"Server {self.server_id}: Starting election (term {self.current_term})")
            
            # In real implementation, this would be async RPC calls
            return votes >= needed_votes
```

### Byzantine Fault Tolerance
```python
class ByzantineNode:
    """Node in Byzantine Fault Tolerant system"""
    
    def __init__(self, node_id: int, total_nodes: int):
        self.node_id = node_id
        self.total_nodes = total_nodes
        self.max_faulty = (total_nodes - 1) // 3
        self.messages = {}
    
    def can_tolerate_faults(self) -> bool:
        """Check if system can tolerate Byzantine faults"""
        # Need at least 3f + 1 nodes to tolerate f faulty nodes
        return self.total_nodes >= 3 * self.max_faulty + 1
    
    def get_consensus_threshold(self) -> int:
        """Minimum votes needed for consensus"""
        return self.total_nodes - self.max_faulty
    
    def validate_message(self, message: Dict, sender: int) -> bool:
        """Validate message integrity"""
        # In real BFT, would verify cryptographic signatures
        return 'value' in message and 'timestamp' in message
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Consistency Models

### Eventual Consistency Pattern
```python
from datetime import datetime, timedelta
from collections import defaultdict

class EventuallyConsistentStore:
    """Eventually consistent distributed key-value store"""
    
    def __init__(self, replica_id: str):
        self.replica_id = replica_id
        self.data = {}  # Current state
        self.version_vectors = defaultdict(dict)  # Vector clocks
        self.pending_updates = []  # Updates not yet replicated
    
    def write(self, key: str, value: str, version_vector: Dict = None) -> str:
        """Write value locally"""
        if version_vector is None:
            version_vector = {self.replica_id: 1}
        else:
            version_vector[self.replica_id] = version_vector.get(self.replica_id, 0) + 1
        
        self.data[key] = value
        self.version_vectors[key] = version_vector
        
        # Track for replication
        self.pending_updates.append({
            'key': key,
            'value': value,
            'version_vector': version_vector,
            'timestamp': datetime.now()
        })
        
        return str(version_vector)
    
    def read(self, key: str):
        """Read value (may be stale)"""
        return self.data.get(key)
    
    def replicate_to(self, other_replica: 'EventuallyConsistentStore'):
        """Replicate pending updates to other replica"""
        for update in self.pending_updates:
            other_replica.merge_update(update)
        self.pending_updates = []
    
    def merge_update(self, update: Dict):
        """Merge update from another replica"""
        key = update['key']
        incoming_vector = update['version_vector']
        
        # Check for causality
        if self._is_concurrent(key, incoming_vector):
            # Conflict! Use application-specific merge strategy
            self._resolve_conflict(key, update['value'], incoming_vector)
        elif self._happened_before(key, incoming_vector):
            # Update is newer, apply it
            self.data[key] = update['value']
            self.version_vectors[key] = incoming_vector
    
    def _happened_before(self, key: str, incoming_vector: Dict) -> bool:
        """Check if incoming update happened after current state"""
        current_vector = self.version_vectors.get(key, {})
        
        for replica, version in incoming_vector.items():
            if version <= current_vector.get(replica, 0):
                return False
        
        return True
    
    def _is_concurrent(self, key: str, incoming_vector: Dict) -> bool:
        """Check if updates are concurrent (no causal relationship)"""
        current_vector = self.version_vectors.get(key, {})
        
        happens_before = all(
            incoming_vector.get(r, 0) >= current_vector.get(r, 0)
            for r in set(list(incoming_vector.keys()) + list(current_vector.keys()))
        )
        
        return not happens_before
    
    def _resolve_conflict(self, key: str, value: str, vector: Dict):
        """Resolve conflicting updates"""
        # Simple strategy: last-write-wins
        if vector.get(self.replica_id, 0) > self.version_vectors[key].get(self.replica_id, 0):
            self.data[key] = value
            self.version_vectors[key] = vector
```

### Quorum-Based Consistency
```python
class QuorumConsistencyStore:
    """Quorum-based consistency for distributed writes"""
    
    def __init__(self, replicas: List[str], quorum_size: int):
        self.replicas = replicas
        self.quorum_size = quorum_size
        self.data = {}
        self.version = {}
    
    def write_with_quorum(self, key: str, value: str) -> bool:
        """Write to quorum of replicas"""
        current_version = self.version.get(key, 0)
        new_version = current_version + 1
        
        write_count = 0
        for replica in self.replicas:
            if self._write_to_replica(replica, key, value, new_version):
                write_count += 1
            
            if write_count >= self.quorum_size:
                self.data[key] = value
                self.version[key] = new_version
                return True
        
        return False
    
    def read_with_quorum(self, key: str) -> tuple:
        """Read from quorum of replicas"""
        values = []
        
        for replica in self.replicas:
            value, version = self._read_from_replica(replica, key)
            if value is not None:
                values.append((value, version))
            
            if len(values) >= self.quorum_size:
                # Return value with highest version
                values.sort(key=lambda x: x[1], reverse=True)
                return values[0][0]
        
        return None
    
    def _write_to_replica(self, replica: str, key: str, value: str, version: int) -> bool:
        """Write to individual replica"""
        # In real implementation, would be RPC call
        return True
    
    def _read_from_replica(self, replica: str, key: str) -> tuple:
        """Read from individual replica"""
        # In real implementation, would be RPC call
        return (self.data.get(key), self.version.get(key, 0))
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Split-Brain Prevention & Leader Election

### Leader Election with Heartbeats
```python
class LeaderElectionCoordinator:
    """Prevent split-brain with leader heartbeats"""
    
    def __init__(self, node_id: str, all_nodes: List[str]):
        self.node_id = node_id
        self.all_nodes = all_nodes
        self.is_leader = False
        self.current_leader = None
        self.heartbeat_timeout = 5.0
        self.last_heartbeat_time = 0
    
    def receive_heartbeat(self, leader_id: str, term: int):
        """Receive leader heartbeat"""
        if leader_id != self.current_leader:
            self.current_leader = leader_id
            self.is_leader = False
        
        self.last_heartbeat_time = time.time()
    
    def check_leader_alive(self) -> bool:
        """Check if current leader is alive"""
        if not self.is_leader:
            elapsed = time.time() - self.last_heartbeat_time
            return elapsed < self.heartbeat_timeout
        return True
    
    def attempt_leadership(self) -> bool:
        """Attempt to become leader if current leader dead"""
        if not self.check_leader_alive():
            # Current leader presumed dead
            # Run leader election
            self.is_leader = self._win_election()
            if self.is_leader:
                self.current_leader = self.node_id
                return True
        
        return False
    
    def _win_election(self) -> bool:
        """Simulate winning leader election"""
        # Vote for self
        votes = 1
        required_votes = len(self.all_nodes) // 2 + 1
        
        # Request votes from other nodes
        for node in self.all_nodes:
            if node != self.node_id:
                if self._request_vote_from(node):
                    votes += 1
        
        return votes >= required_votes
    
    def _request_vote_from(self, node: str) -> bool:
        """Request vote from node"""
        return True  # In real system, would be RPC
    
    def send_heartbeats(self):
        """Send heartbeats to maintain leadership"""
        if self.is_leader:
            for node in self.all_nodes:
                if node != self.node_id:
                    self._send_heartbeat_to(node)
    
    def _send_heartbeat_to(self, node: str):
        """Send heartbeat to node"""
        pass  # In real system, would be RPC
```

### Failure Detection
```python
class FailureDetector:
    """Detect node failures with adaptive timeouts"""
    
    def __init__(self, nodes: List[str]):
        self.nodes = nodes
        self.heartbeats = {node: time.time() for node in nodes}
        self.suspicious = set()
        self.failed = set()
        self.phi_threshold = 1.5
    
    def record_heartbeat(self, node: str):
        """Record heartbeat from node"""
        self.heartbeats[node] = time.time()
        self.suspicious.discard(node)
    
    def check_suspicion(self, node: str) -> float:
        """Calculate suspicion level (phi accrual)"""
        if node not in self.heartbeats:
            return float('inf')
        
        elapsed = time.time() - self.heartbeats[node]
        phi = -math.log10((1 - elapsed / 1000) / elapsed)
        
        return phi
    
    def update_status(self):
        """Update node status based on suspicion levels"""
        for node in self.nodes:
            phi = self.check_suspicion(node)
            
            if phi > self.phi_threshold:
                self.suspicious.add(node)
            elif phi > self.phi_threshold * 1.5:
                self.failed.add(node)
                self.suspicious.discard(node)
            else:
                self.suspicious.discard(node)
                self.failed.discard(node)
    
    def is_suspected(self, node: str) -> bool:
        """Check if node is suspected failed"""
        return node in self.suspicious
    
    def is_failed(self, node: str) -> bool:
        """Check if node is confirmed failed"""
        return node in self.failed
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Distributed Systems Checklist

✅ **Consensus**
- [ ] Consensus algorithm selected (Raft/Paxos/BFT)
- [ ] Leader election implemented
- [ ] State machine replication working
- [ ] Membership changes handled
- [ ] Recovery from failures tested

✅ **Consistency**
- [ ] Consistency model chosen (strong/eventual/causal)
- [ ] Version vectors or vector clocks implemented
- [ ] Conflict detection working
- [ ] Conflict resolution strategy defined
- [ ] Monitoring for anomalies

✅ **Reliability**
- [ ] Heartbeats implemented
- [ ] Failure detection working
- [ ] Quorum-based operations tested
- [ ] Split-brain prevention verified
- [ ] Recovery from network partitions

✅ **Performance**
- [ ] Replication latency measured
- [ ] Throughput under load tested
- [ ] Scaling properties verified
- [ ] Bottlenecks identified
- [ ] Optimization targets set

✅ **Operations**
- [ ] Monitoring dashboard created
- [ ] Alerts configured for failures
- [ ] Runbooks for common failures written
- [ ] Data consistency validation tools built
- [ ] Disaster recovery procedures documented
</VSCode.Cell>
```